In [0]:
from pyspark.sql.functions import * 
from delta.tables import *

In [0]:
%run /Workspace/Users/narutosha289@gmail.com/Consolidated_pipeline/Setup/Utilities 

In [0]:
print(bronze_schema, silver_schema, gold_schema)

Bronze Silver Gold


In [0]:
dbutils.widgets.text("Catalog","FMCG","Catalog") 
dbutils.widgets.text("Data_Source","orders","Data_Source") 

Catalog = dbutils.widgets.get("Catalog")
Data_Source = dbutils.widgets.get("Data_Source")
BasePath = "s3://sportsbar-us1/orders"
LandingPath = f"{BasePath}/landing/" 
ProcessedPath = f"{BasePath}/processed/"
print(BasePath)
print(LandingPath)
print(ProcessedPath) 

s3://sportsbar-us1/orders
s3://sportsbar-us1/orders/landing/
s3://sportsbar-us1/orders/processed/


### Bronze Processing

In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(LandingPath)
        .withColumn("read_timestamp", current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
) 

# Print the toal rows: 
print("Total Rows:",df.count()) 
df.show()

Total Rows: 670
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FDEC85101603|Wednesday, Decemb...|     789101|  25891301|     92.0|2026-09-16 11:53:...|orders_2025_12_03...|    21899|
|FDEC85101603|Wednesday, Decemb...|     789101|  25891502|    232.0|2026-09-16 11:53:...|orders_2025_12_03...|    21899|
|FDEC85101603|          03/12/2025|     789101|  25891402|    478.0|2026-09-16 11:53:...|orders_2025_12_03...|    21899|
|FDEC85101603|Wednesday, Decemb...|     789101|  25891201|    349.0|2026-09-16 11:53:...|orders_2025_12_03...|    21899|
|FDEC85101603|Wednesday, Decemb...|     789101|  25891602|     62.0|2026-09-16 11:53:...|orders_2025_12_03...|    21899|
|FDEC85101603|We

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(f"{Catalog}.{bronze_schema}.{Data_Source}")

### Staging table for the incremental data

In [0]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{Catalog}.{bronze_schema}.staging_{Data_Source}")

### Moving the files for landing to the processed

In [0]:
files = dbutils.fs.ls(LandingPath)
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{ProcessedPath}/{file_info.name}",
        True
    )

### Silver Processing

In [0]:
df_orders = spark.read.table(f"{Catalog}.{bronze_schema}.{Data_Source}")
df_orders.show()
# Keep only the rows where order_qty is present
df_orders = df_orders.filter(df_orders["order_qty"].isNotNull()) 
df_orders.show()
# Clean the customer_id keep only the numeric values other values to 0 and cast to string
df_orders = df_orders.withColumn("customer_id", when(df_orders["customer_id"].rlike("^[0-9]+$"), df_orders["customer_id"]).otherwise(99999).cast("string"))
df_orders.show()
# Replace the weekday name from the order placement day: 
df_orders = df_orders.withColumn("order_placement_date", regexp_replace(df_orders["order_placement_date"], r"^[A-Za-z]+,\s*", "")) 
df_orders.show()
# Parasing order placement date from multiple formats: 
date_formats = ["yyyy/MM/dd", "dd/MM/yyyy", "yyyy-MM-dd", "dd-MM-yyyy","MMMM dd, yyyy"] 
df_orders = df_orders.withColumn("order_placement_date", coalesce(*[try_to_date(df_orders["order_placement_date"], fmt) for fmt in date_formats]))
df_orders.show()
# Drop duplicates: 
df_orders = df_orders.dropDuplicates(["customer_id","order_id","order_placement_date","order_qty","product_id"])
df_orders.show()
df_orders.printSchema()
# Converting product_id into string: 
df_orders = df_orders.withColumn("product_id", df_orders["product_id"].cast("string"))
df_orders.printSchema()
# Get the max amd min order placement date: 
df_orders.agg(min("order_placement_date").alias("min_date"), max("order_placement_date").alias("max_date")).show()

+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|    order_id|order_placement_date|customer_id|product_id|order_qty|      read_timestamp|           file_name|file_size|
+------------+--------------------+-----------+----------+---------+--------------------+--------------------+---------+
|FDEC83401502|Tuesday, December...|     789401|  25891203|    256.0|2026-09-16 11:53:...|orders_2025_12_02...|    19621|
|FDEC83401502|Tuesday, December...|     789401|  25891502|    218.0|2026-09-16 11:53:...|orders_2025_12_02...|    19621|
|FDEC83401502|Tuesday, December...|     789401|  25891403|    280.0|2026-09-16 11:53:...|orders_2025_12_02...|    19621|
|FDEC83401502|Tuesday, December...|     789401|  25891201|    262.0|2026-09-16 11:53:...|orders_2025_12_02...|    19621|
|FDEC83401502|Tuesday, December...|     789401|  25891203|    256.0|2026-09-16 11:53:...|orders_2025_12_02...|    19621|
|FDEC83401502|Tuesday, December.

In [0]:
# Join with the products table to get the product code: 
df_products = spark.read.table(f"{Catalog}.{silver_schema}.products")
df_joined = df_orders.join(df_products, df_orders["product_id"] == df_products["product_id"],'inner').select(df_orders["*"], df_products["product_code"])
df_joined.show(truncate=False)

+------------+--------------------+-----------+----------+---------+--------------------------+---------------------+---------+----------------------------------------------------------------+
|order_id    |order_placement_date|customer_id|product_id|order_qty|read_timestamp            |file_name            |file_size|product_code                                                    |
+------------+--------------------+-----------+----------+---------+--------------------------+---------------------+---------+----------------------------------------------------------------+
|FJUL32721603|2025-07-01          |99999      |25891202  |169.0    |2026-09-11 15:48:15.649157|orders_2025_07_01.csv|20744    |0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|
|FJUL33501401|2025-07-01          |789501     |25891303  |57.0     |2026-09-11 15:48:15.649157|orders_2025_07_01.csv|20744    |c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|
|FJUL34903603|2025-07-01          |

In [0]:
df_joined.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable(f"{Catalog}.{silver_schema}.{Data_Source}")

In [0]:
df_joined.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("overwrite") \
 .saveAsTable(f"{Catalog}.{silver_schema}.staging_{Data_Source}")

### Gold Processing

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {Catalog}.{silver_schema}.staging_{Data_Source};")
df_gold.show(2)

+------------+----------+-------------+--------------------+----------+-------------+
|    order_id|      date|customer_code|        product_code|product_id|sold_quantity|
+------------+----------+-------------+--------------------+----------+-------------+
|FJUL32721603|2025-07-01|        99999|0cb7b2f42657b625f...|  25891202|        169.0|
|FJUL33501401|2025-07-01|       789501|c68834ceaff15846b...|  25891303|         57.0|
+------------+----------+-------------+--------------------+----------+-------------+
only showing top 2 rows


In [0]:
df_gold.write.format("delta").option(
    "delta.enableChangeDataFeed", "true"
).option("mergeSchema", "true").mode("overwrite").saveAsTable(f"{Catalog}.{gold_schema}.{Data_Source}")

### Merging with the parent table:

In [0]:
# df_child = your incremental daily rows

df_child =  spark.sql(f"SELECT order_placement_date as date FROM {Catalog}.{silver_schema}.staging_{Data_Source}")

incremental_month_df = df_child.select(
    trunc("date", "MM").alias("start_month")
).distinct()

incremental_month_df.show()

incremental_month_df.createOrReplaceTempView("incremental_months")

+-----------+
|start_month|
+-----------+
| 2025-08-01|
| 2025-10-01|
| 2025-12-01|
| 2025-09-01|
| 2025-11-01|
| 2025-07-01|
+-----------+



In [0]:
monthly_table = spark.sql(f"""
    SELECT date, product_code, customer_code, sold_quantity
    FROM {Catalog}.{gold_schema}.sb_fact_orders sbf
    INNER JOIN incremental_months m
        ON trunc(sbf.date, 'MM') = m.start_month
""")

print("Total Rows: ", monthly_table.count())
monthly_table.show(10)

Total Rows:  40811
+----------+--------------------+-------------+-------------+
|      date|        product_code|customer_code|sold_quantity|
+----------+--------------------+-------------+-------------+
|2025-07-01|0cb7b2f42657b625f...|        99999|        169.0|
|2025-07-01|c68834ceaff15846b...|       789501|         57.0|
|2025-07-01|ee1f7df9cf660ef02...|       789903|        136.0|
|2025-07-01|c68834ceaff15846b...|       789203|         64.0|
|2025-07-01|889c67757ece9c973...|       789421|         99.0|
|2025-07-01|778c2a7aa27bfdb21...|       789102|        154.0|
|2025-07-01|3cab59f0592428527...|       789103|         67.0|
|2025-07-01|0cb7b2f42657b625f...|       789401|        173.0|
|2025-07-01|778c2a7aa27bfdb21...|       789102|         63.0|
|2025-07-01|716fa4e54b7894c91...|       789520|        146.0|
+----------+--------------------+-------------+-------------+
only showing top 10 rows


In [0]:
df_monthly_recalc = (
    monthly_table
    .withColumn("month_start", trunc("date", "MM"))
    .groupBy("month_start", "product_code", "customer_code")
    .agg(sum("sold_quantity").alias("sold_quantity"))
    .withColumnRenamed("month_start", "date")   # month_start → date = first of month
)

df_monthly_recalc.show(10, truncate=False)

+----------+----------------------------------------------------------------+-------------+-------------+
|date      |product_code                                                    |customer_code|sold_quantity|
+----------+----------------------------------------------------------------+-------------+-------------+
|2025-07-01|ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae|789903       |1924.0       |
|2025-07-01|889c67757ece9c973791dfbc2d47b026a3342cc7255e47a3170329d158e897c2|789421       |3318.0       |
|2025-07-01|0cb7b2f42657b625f754e833aa1cf6a967be26f17415f5342302ebb0e90c8a28|789401       |2103.0       |
|2025-07-01|77b6f538a9d0e0cf845db5c2cbecec46fdd30303b501e06f64baf1d4dc0e66f9|789202       |5275.0       |
|2025-07-01|c68834ceaff15846bc1892c2185dc4e4f471d64fe3796b1a8ecc39a5a48c614f|789202       |848.0        |
|2025-07-01|ee1f7df9cf660ef02c33037d8d6eb94cbefe8e7b84c306e9387f09b0cae0abae|789203       |1936.0       |
|2025-07-01|e92c739a8d78cd6cbe954648c2f9dd75ed

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{Catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly_recalc.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

### Cleaning the staging table: 

In [0]:
%sql
DROP TABLE fmcg.bronze.staging_orders;

In [0]:
%sql
DROP TABLE fmcg.silver.staging_orders;